# PerioVision AI: train the tooth-detector fine-tune on Aga Khan films (model D) on a free Colab GPU

`Runtime` → `Change runtime type` → **T4 GPU** → Save, then `Runtime` → **Run all** and allow Google Drive.
About 30-45 minutes on a free T4. **First** upload `backend/weights/dental_yolov8n.pt` to `MyDrive/PerioVision/current/` in THIS Google account. If Colab disconnects, Run all again: training resumes from Drive.

Results go to `MyDrive/PerioVision/export_panoramic/` in this account; download that folder and give it to Claude.

In [ ]:
# 0. Settings (defaults chosen for a free T4 GPU)
# Which models to train in this session. Finished models are skipped anyway, so a short session can do one at a time.
RUN = {"A": False, "B": False, "C": False, "D": True}   # D (tooth-detector fine-tune) can also be trained on a CPU
ARCH        = "convnext_tiny"   # backbone for A and B: resnet18 | resnet50 | efficientnet_b3 | convnext_tiny
SIZE        = (1024, 512)       # panoramic film resized to W x H for A and B
EPOCHS_A, EPOCHS_B = 12, 30
BATCH_A, BATCH_B   = 16, 16
# C: per-tooth bone-loss detector. "yolo11s" at 1024 fits one free session (~1-1.5 h on a T4);
# with more GPU time use "yolo11m.pt", 1280, 150 for the most accuracy.
YOLO_C, IMGSZ_C, EPOCHS_C, PATIENCE_C = "yolo11s.pt", 1024, 60, 15
IMGSZ_D, EPOCHS_D = 1280, 40

In [ ]:
# 1. GPU + Google Drive
import subprocess, os, glob, json, shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError("NO GPU attached. Runtime > Change runtime type > T4 GPU > Save, then Run all again. "
                       "If Colab says 'Cannot connect to GPU backend', this account's free GPU time is used up "
                       "for today: try again later or use another Google account.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
EXPORT = f"{ROOT}/export_panoramic"
for d in ("runs", "current", "export_panoramic"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

In [ ]:
# 2. Same Ultralytics version as the PerioVision backend, plus the Google-Drive downloader
!pip -q install "ultralytics==8.4.21" gdown
import ultralytics; ultralytics.checks()

In [ ]:
%%writefile /content/convert_aku_labelme.py
"""Aga Khan University OPG dataset (Zenodo 10.5281/zenodo.10538750, CC BY 4.0) -> YOLO detect, FDI classes.

LabelMe polygons (crown + root) with the FDI number in `group_id` become boxes whose class index matches
the PerioVision tooth detector exactly: 11-18 -> 0-7, 21-28 -> 8-15, 31-38 -> 16-23, 41-48 -> 24-31.
Films are split 70 / 15 / 15 (seeded) so no film appears in two splits; the test split is kept for the
before / after comparison and never used for training. All three folders are used (folder 2 spells its label
folder "annnotations"; versions before 2026-10-04 missed it, so the first fine-tune saw folders 1 and 3 only).

Usage:  python convert_aku_labelme.py --src <.../dataset> --out <folder>
"""
import argparse
import glob
import json
import os
import random
import shutil

FDI = [f"{q}{n}" for q in (1, 2, 3, 4) for n in range(1, 9)]


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    src = os.path.expanduser(args.src)
    # folder 2 of the published archive spells its label folder "annnotations" (three n)
    anns = sorted(glob.glob(os.path.join(src, "*", "annotations", "*.json")) +
                  glob.glob(os.path.join(src, "*", "annnotations", "*.json")))
    random.Random(0).shuffle(anns)  # audit-ok: seeded film-level split
    n = len(anns)
    counts = {"train": 0, "val": 0, "test": 0}
    for i, ann in enumerate(anns):
        split = "train" if i < 0.7 * n else "val" if i < 0.85 * n else "test"
        d = json.load(open(ann, encoding="utf-8"))
        stem = os.path.splitext(os.path.basename(ann))[0]
        imgs = glob.glob(os.path.join(os.path.dirname(ann), "..", "OPGs", stem + ".*"))
        if not imgs:
            continue
        w, h = d["imageWidth"], d["imageHeight"]
        lines = []
        for s in d["shapes"]:
            fdi = str(s.get("group_id"))
            if fdi not in FDI or s.get("shape_type") != "polygon":
                continue
            xs, ys = [p[0] for p in s["points"]], [p[1] for p in s["points"]]
            x1, x2, y1, y2 = max(0, min(xs)), min(w, max(xs)), max(0, min(ys)), min(h, max(ys))
            lines.append(f"{FDI.index(fdi)} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        if not lines:
            continue
        name = f"{os.path.basename(os.path.dirname(os.path.dirname(ann))).replace(' ', '_')}_{stem}"
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(args.out, sub, split), exist_ok=True)
        shutil.copy(imgs[0], os.path.join(args.out, "images", split, name + os.path.splitext(imgs[0])[1]))
        with open(os.path.join(args.out, "labels", split, name + ".txt"), "w") as f:
            f.write("\n".join(lines) + "\n")
        counts[split] += 1
    with open(os.path.join(args.out, "aku.yaml"), "w") as f:
        f.write(f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                + "".join(f"  {i}: '{c}'\n" for i, c in enumerate(FDI)))
    print(counts)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# 3. Download the four datasets (skipped if already present)
import urllib.request, zipfile
D = "/content/data"; os.makedirs(D, exist_ok=True)
def fetch(url, path):
    if not os.path.exists(path):
        print("downloading", url); urllib.request.urlretrieve(url, path)
    return path

need_A = RUN["A"] and not os.path.exists(f"{EXPORT}/panoramic_screen_metrics.json")
need_B = RUN["B"] and not os.path.exists(f"{EXPORT}/panoramic_severity_metrics.json")
need_C = RUN["C"] and not os.path.exists(f"{EXPORT}/pdcnn_boneloss_metrics.json")
need_D = RUN["D"] and not os.path.exists(f"{EXPORT}/aku_detector_finetune_metrics.json")
print("still to train:", [k for k, v in (("A", need_A), ("B", need_B), ("C", need_C), ("D", need_D)) if v])

# MM-OPG (Hugging Face, Apache-2.0) - only needed for A
if need_A:
    os.makedirs(f"{D}/mmopg", exist_ok=True)
    HF = "https://huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations/resolve/main"
    fetch(f"{HF}/MM-OPG-Train-Raw.json", f"{D}/mmopg/mmopg_train.json")
    fetch(f"{HF}/MM-OPG-Test.json", f"{D}/mmopg/mmopg_test.json")
    fetch(f"{HF}/images_resized_public.zip", f"{D}/mmopg/images.zip")

# BRAR (figshare, CC BY 4.0) - only needed for B
if need_B and not os.path.exists(f"{D}/brar/meta_data.csv"):
    zipfile.ZipFile(fetch("https://ndownloader.figshare.com/files/58062268", f"{D}/brar.zip")).extractall(f"{D}/brar")
    inner = glob.glob(f"{D}/brar/**/meta_data.csv", recursive=True)[0]
    if os.path.dirname(inner) != f"{D}/brar":
        for p in os.listdir(os.path.dirname(inner)): shutil.move(os.path.join(os.path.dirname(inner), p), f"{D}/brar")

# Aga Khan OPG (Zenodo, CC BY 4.0) - only needed for D
if need_D and not glob.glob(f"{D}/aku/**/annotations", recursive=True):
    zipfile.ZipFile(fetch("https://zenodo.org/api/records/10538750/files/Niihhaa/Dataset-v1.zip/content", f"{D}/aku.zip")).extractall(f"{D}/aku")

# PDCNN (public Google Drive folder) - only needed for C
if need_C:
    if not glob.glob(f"{D}/pdcnn/**/*.json", recursive=True):
        !gdown --folder "https://drive.google.com/drive/folders/18qUxeRPHPcCQT9o8AgV5400f05Fu3ISW" -O {D}/pdcnn
    for z in glob.glob(f"{D}/pdcnn/**/*.zip", recursive=True):
        if not os.path.isdir(z[:-4]): zipfile.ZipFile(z).extractall(z[:-4])
    print("PDCNN annotation files:", glob.glob(f"{D}/pdcnn/**/*.json", recursive=True))
print(subprocess.run("du -sh " + D + "/*", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 7. Model D: fine-tune the current tooth detector on Aga Khan; keep it only if the held-out test improves
from ultralytics import YOLO
CUR = f"{ROOT}/current/dental_yolov8n.pt"
if not need_D:
    print("Model D already trained or switched off in RUN - skipping")
elif not os.path.exists(CUR):
    print("SKIPPED: upload backend/weights/dental_yolov8n.pt to MyDrive/PerioVision/current/ to run step D")
else:
    aku_src = os.path.dirname(glob.glob(f"{D}/aku/**/folder 1", recursive=True)[0])
    if not os.path.exists("/content/aku_yolo/aku.yaml"):
        !python /content/convert_aku_labelme.py --src "{aku_src}" --out /content/aku_yolo
    before = YOLO(CUR).val(data="/content/aku_yolo/aku.yaml", split="test", imgsz=IMGSZ_D, project=f"{ROOT}/runs", name="aku_before", exist_ok=True)
    run_d = f"{ROOT}/runs/aku_finetune"
    if not os.path.exists(f"{run_d}/weights/best.pt"):
        YOLO(CUR).train(data="/content/aku_yolo/aku.yaml", imgsz=IMGSZ_D, epochs=EPOCHS_D, patience=20, batch=-1, lr0=0.001,
                        freeze=10, fliplr=0.0, project=f"{ROOT}/runs", name="aku_finetune", exist_ok=True, seed=0)
    after = YOLO(f"{run_d}/weights/best.pt").val(data="/content/aku_yolo/aku.yaml", split="test", imgsz=IMGSZ_D, project=f"{ROOT}/runs", name="aku_after", exist_ok=True)
    md_ = {"test_mAP50_before": round(float(before.box.map50), 4), "test_mAP50_after": round(float(after.box.map50), 4),
           "test_mAP50_95_before": round(float(before.box.map), 4), "test_mAP50_95_after": round(float(after.box.map), 4),
           "note": "fliplr=0 because mirroring would swap left / right FDI numbers"}
    md_["adopted"] = md_["test_mAP50_95_after"] > md_["test_mAP50_95_before"]
    json.dump(md_, open(f"{EXPORT}/aku_detector_finetune_metrics.json", "w"), indent=2)
    if md_["adopted"]:
        shutil.copy(f"{run_d}/weights/best.pt", f"{EXPORT}/dental_yolov8n_aku.pt")
    print(json.dumps(md_, indent=2))

In [ ]:
# 8. Summary of everything in the export folder
for f in sorted(glob.glob(f"{EXPORT}/*")):
    print(f"{os.path.basename(f):45s} {os.path.getsize(f)/1e6:8.1f} MB")
for f in sorted(glob.glob(f"{EXPORT}/*metrics.json")):
    print("\n==", os.path.basename(f)); print(open(f).read())